In [7]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [8]:
df = yf.download("RELIANCE.NS", start="2011-01-01", end="2024-01-01",auto_adjust=True)
df.head()
reliance = yf.Ticker("RELIANCE.NS")
df = df.droplevel("Ticker", axis=1)
train = df[:"2020-12-31"]  # Jan 2011 – Dec 2020 (10 years)
test  = df["2021-01-01":]  # Jan 2021 – Jan 2024  (3 years)

[*********************100%***********************]  1 of 1 completed


In [9]:
def create_target(data, horizon=5):
    # Binary target: 1 if Close(T+horizon) > Close(T), else 0.
    if isinstance(data.columns, pd.MultiIndex):
        data = data.droplevel("Ticker", axis=1)
    close = data["Close"]
    future_close = close.shift(-horizon)
    target = (future_close > close).astype(int)
    target.name = "target"
    return target


In [10]:
def build_features(data):
    """
    Engineer features from OHLCV data for logistic regression.
    Returns a DataFrame of features aligned with the original index.
    """
    # Flatten MultiIndex columns if present (yfinance returns multi-level cols)
    if isinstance(data.columns, pd.MultiIndex):
        data = data.droplevel("Ticker", axis=1)

    close = data["Close"]
    volume = data["Volume"]

    feat = pd.DataFrame(index=data.index)

    # Price returns
    feat["ret_1"]  = close.pct_change(1)    # 1-day return
    feat["ret_5"]  = close.pct_change(5)    # 5-day return
    feat["ret_10"] = close.pct_change(10)   # 10-day return
    feat["ret_20"] = close.pct_change(20)   # 20-day return

    # Moving averages (ratio to current price)
    feat["ma5_ratio"]  = close / close.rolling(5).mean()
    feat["ma10_ratio"] = close / close.rolling(10).mean()
    feat["ma20_ratio"] = close / close.rolling(20).mean()
    feat["ma50_ratio"] = close / close.rolling(50).mean()

    # Volatility (rolling std of returns)
    feat["vol_5"]  = close.pct_change().rolling(5).std()
    feat["vol_20"] = close.pct_change().rolling(20).std()

    # RSI (14-day)
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss
    feat["rsi_14"] = 100 - (100 / (1 + rs))

    # Volume change
    feat["vol_change_1"] = volume.pct_change(1)
    feat["vol_change_5"] = volume.pct_change(5)

    # Rolling Z-scores (how far price is from recent average, in std devs)
    feat["zscore_20"] = (close - close.rolling(20).mean()) / close.rolling(20).std()
    feat["zscore_50"] = (close - close.rolling(50).mean()) / close.rolling(50).std()

    # Replace any inf values with NaN (e.g. from zero-volume days)
    feat.replace([np.inf, -np.inf], np.nan, inplace=True)

    return feat

In [11]:
def run_logistic_regression(train_df, test_df, horizon=5):
    """
    Train a Logistic Regression model on train_df and evaluate on test_df.
    Predicts whether Close(T+5) > Close(T).
    """
    # Build features & target for both sets
    X_train = build_features(train_df)
    y_train = create_target(train_df, horizon)

    X_test = build_features(test_df)
    y_test = create_target(test_df, horizon)

    # Align and drop rows with NaN (from rolling windows / shift)
    train_combined = pd.concat([X_train, y_train], axis=1).dropna()
    test_combined  = pd.concat([X_test, y_test], axis=1).dropna()

    X_train_clean = train_combined.drop("target", axis=1)
    y_train_clean = train_combined["target"]
    X_test_clean  = test_combined.drop("target", axis=1)
    y_test_clean  = test_combined["target"]

    # Standardize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_clean)
    X_test_scaled  = scaler.transform(X_test_clean)

    # Train Logistic Regression
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train_scaled, y_train_clean)

    # Predict
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred  = model.predict(X_test_scaled)

    # ---- Results ----
    print("\n" + "=" * 60)
    print("  LOGISTIC REGRESSION – Predict Close(T+5) > Close(T)")
    print("=" * 60)

    print(f"\n--- Train Set Results ---")
    print(f"Accuracy: {accuracy_score(y_train_clean, y_train_pred):.4f}")
    print(classification_report(y_train_clean, y_train_pred, target_names=["Down/Flat", "Up"]))

    print(f"--- Test Set Results ---")
    print(f"Accuracy: {accuracy_score(y_test_clean, y_test_pred):.4f}")
    print(classification_report(y_test_clean, y_test_pred, target_names=["Down/Flat", "Up"]))

    print("Confusion Matrix (Test):")
    print(confusion_matrix(y_test_clean, y_test_pred))

    # Feature importance (coefficient magnitudes)
    feature_names = X_train_clean.columns
    coefs = pd.Series(model.coef_[0], index=feature_names).sort_values(key=abs, ascending=False)
    print("\nTop Feature Coefficients:")
    print(coefs.to_string())

    return model, scaler

In [12]:
model, scaler = run_logistic_regression(train, test, horizon=5)


  LOGISTIC REGRESSION – Predict Close(T+5) > Close(T)

--- Train Set Results ---
Accuracy: 0.5577
              precision    recall  f1-score   support

   Down/Flat       0.54      0.31      0.39      1120
          Up       0.56      0.77      0.65      1290

    accuracy                           0.56      2410
   macro avg       0.55      0.54      0.52      2410
weighted avg       0.55      0.56      0.53      2410

--- Test Set Results ---
Accuracy: 0.5202
              precision    recall  f1-score   support

   Down/Flat       0.48      0.31      0.37       324
          Up       0.54      0.71      0.61       368

    accuracy                           0.52       692
   macro avg       0.51      0.51      0.49       692
weighted avg       0.51      0.52      0.50       692

Confusion Matrix (Test):
[[ 99 225]
 [107 261]]

Top Feature Coefficients:
ma10_ratio     -0.306072
ma20_ratio      0.223592
vol_5           0.132054
ret_10          0.130862
zscore_50      -0.119250
ma5_r